In [1]:
import oceanbench

oceanbench.__version__

'0.0.2'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# Open GLONET forecast sample with xarray
from datetime import datetime
import xarray
from oceanbench.core.interpolate import interpolate_1deg
challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [
        "https://minio.dive.edito.eu/project-oceanbench/public/glonet_full_2024/20240103.zarr",
        "https://minio.dive.edito.eu/project-oceanbench/public/glonet_full_2024/20240110.zarr",
    ],
    engine="zarr",
    preprocess=lambda dataset: dataset.rename({"time": "lead_day_index"}).assign({"lead_day_index": range(10)}),
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=True,
).assign(
    {
        "first_day_datetime": [
            datetime.fromisoformat("2024-01-03"),
            datetime.fromisoformat("2024-01-10"),
        ]
    }
)
challenger_dataset = interpolate_1deg(challenger_dataset)
challenger_dataset



<xarray.Dataset> Size: 881MB
Dimensions:             (first_day_datetime: 2, lead_day_index: 10, depth: 21,
                         latitude: 180, longitude: 360)
Coordinates:
  * depth               (depth) float32 84B 0.494 47.37 ... 4.833e+03 5.275e+03
  * lead_day_index      (lead_day_index) int64 80B 0 1 2 3 4 5 6 7 8 9
  * first_day_datetime  (first_day_datetime) datetime64[ns] 16B 2024-01-03 20...
  * latitude            (latitude) float64 1kB -89.5 -88.5 -87.5 ... 88.5 89.5
  * longitude           (longitude) float64 3kB -179.5 -178.5 ... 178.5 179.5
Data variables:
    so                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 218MB dask.array<chunksize=(1, 10, 1, 180, 360), meta=np.ndarray>
    thetao              (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 218MB dask.array<chunksize=(1, 10, 1, 180, 360), meta=np.ndarray>
    uo                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 218MB dask.array<chunksize=(1, 10, 1, 180, 360), meta=np.ndarray>
    vo                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 218MB dask.array<chunksize=(1, 10, 1, 180, 360), meta=np.ndarray>
    zos                 (first_day_datetime, lead_day_index, latitude, longitude) float64 10MB dask.array<chunksize=(1, 10, 180, 360), meta=np.ndarray>
Attributes:
    regrid_method:  bilinear

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [3]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Surface height,0.068839,0.073657,0.070511,0.074330,0.074626,0.080343,0.079516,0.083759,0.082740,0.088912
Surface temperature,0.626146,0.640253,0.669531,0.679004,0.724654,0.738378,0.789644,0.799838,0.845998,0.855692
Surface salinity,0.675697,0.673403,0.675221,0.672760,0.669974,0.666340,0.665058,0.664155,0.666172,0.665457
Surface northward velocity,0.124271,0.122609,0.123297,0.123603,0.125209,0.126580,0.129877,0.132186,0.136998,0.137461
Surface eastward velocity,0.124240,0.123775,0.125366,0.126338,0.129091,0.129721,0.133317,0.134709,0.138357,0.139857
50m temperature,0.943730,0.953655,0.971085,0.988293,1.020550,1.057144,1.096793,1.153087,1.185156,1.248715
50m salinity,0.348172,0.347124,0.354332,0.354112,0.358376,0.358829,0.363182,0.364317,0.368755,0.369989
50m northward velocity,0.108935,0.108866,0.107223,0.107654,0.107780,0.110010,0.113079,0.116871,0.119323,0.120542
50m eastward velocity,0.110337,0.109822,0.109378,0.110205,0.111231,0.113171,0.116007,0.119487,0.122836,0.126496
200m temperature,0.827891,0.834058,0.836950,0.841294,0.839004,0.847537,0.851307,0.865344,0.872938,0.886207


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Mixed layer depth,38.877853,39.680756,42.396683,43.25692,45.84166,46.959801,48.683277,50.034492,50.925659,52.825287


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Northward geostrophic velocity,0.054896,0.056216,0.059535,0.062112,0.064237,0.068141,0.070456,0.076220,0.077217,0.084376
Eastward geostrophic velocity,0.051826,0.056432,0.060964,0.062050,0.063164,0.067745,0.073030,0.076103,0.077824,0.085527


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(challenger_dataset)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Surface Lagrangian trajectory deviation (km),9.70379,17.764771,24.914489,31.577176,37.954062,44.223183,50.423489,56.558537


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Surface height,0.022215,0.025949,0.027534,0.033707,0.041710,0.047824,0.053222,0.059289,0.061345,0.065171
Surface temperature,0.328126,0.359627,0.454374,0.477620,0.569905,0.591176,0.669199,0.689965,0.749652,0.765774
Surface salinity,0.116509,0.119577,0.174601,0.181153,0.227020,0.235828,0.267669,0.272675,0.297746,0.300315
Surface northward velocity,0.069158,0.070534,0.071884,0.081973,0.088986,0.098463,0.108488,0.117545,0.126458,0.131716
Surface eastward velocity,0.065499,0.069421,0.072516,0.081868,0.091440,0.100495,0.111252,0.119476,0.126307,0.130811
50m temperature,0.677056,0.678493,0.704412,0.727812,0.791045,0.835341,0.902651,0.966708,1.021159,1.087757
50m salinity,0.088248,0.089519,0.123603,0.126815,0.148457,0.152623,0.168100,0.173233,0.185322,0.188497
50m northward velocity,0.048454,0.050463,0.054867,0.059911,0.066059,0.073513,0.082704,0.091686,0.098802,0.104294
50m eastward velocity,0.046888,0.048348,0.054733,0.060728,0.067622,0.074807,0.083702,0.092977,0.101185,0.108889
200m temperature,0.383957,0.389907,0.440736,0.457228,0.492494,0.526166,0.563495,0.599442,0.627937,0.662338


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [8]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Mixed layer depth,36.834183,37.220116,41.12553,41.516674,45.453224,45.752274,48.631996,49.2705,50.550781,51.234131


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLOR12 analysis

In [9]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Northward geostrophic velocity,0.027203,0.031931,0.039641,0.043674,0.048697,0.055469,0.060165,0.068450,0.070636,0.078661
Eastward geostrophic velocity,0.029858,0.038963,0.044587,0.046500,0.049852,0.057546,0.065898,0.071395,0.071928,0.079585


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [10]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(challenger_dataset)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Surface Lagrangian trajectory deviation (km),4.341065,8.084498,11.844843,15.988134,20.452688,25.475823,30.938677,36.66349
